In [6]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import os

# Đọc lại dữ liệu thô
train = pd.read_csv('../../data/train.csv')
test = pd.read_csv('../../data/test.csv')

# Tách biến mục tiêu (y) và gộp Train + Test để xử lý đồng nhất (X)
y_train = np.log1p(train['SalePrice'])
train.drop(['SalePrice', 'Id'], axis=1, inplace=True)
test_id = test['Id']
test.drop(['Id'], axis=1, inplace=True)

df_all = pd.concat([train, test], axis=0).reset_index(drop=True)

# 1. Xử lý Missing Values
# Các cột phân loại mà 'NA' có nghĩa là 'None' (Không có)
cols_none = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'MasVnrType']
for col in cols_none:
    df_all[col] = df_all[col].fillna('None')

# Các cột số điền bằng 0 (Không có diện tích/số lượng)
cols_zero = ['GarageYrBlt', 'GarageArea', 'GarageCars', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea']
for col in cols_zero:
    df_all[col] = df_all[col].fillna(0)

# LotFrontage điền bằng Median của Neighborhood
df_all['LotFrontage'] = df_all.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))

# Nếu vẫn còn Missing thì điền bằng Median toàn bộ cột
df_all['LotFrontage'] = df_all['LotFrontage'].fillna(df_all['LotFrontage'].median())

# Các cột phân loại còn lại điền bằng Mode (giá trị xuất hiện nhiều nhất)
cols_mode = ['MSZoning', 'Electrical', 'KitchenQual', 'Exterior1st', 'Exterior2nd', 'SaleType']
for col in cols_mode:
    df_all[col] = df_all[col].fillna(df_all[col].mode()[0])

df_all = df_all.drop(['Utilities'], axis=1) # Cột này 99% giống nhau, không có ý nghĩa

# 2. Encoding cho các cột Categorical
# Các cột có thứ tự được mã hóa theo đúng thứ tự trong data_description
ordinal_cols = {
    'ExterQual': {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
    'ExterCond': {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
    'BsmtQual': {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
    'BsmtCond': {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
    'BsmtExposure': {'None': 0, 'No': 1, 'Mn': 2, 'Av': 3, 'Gd': 4},
    'BsmtFinType1': {'None': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6},
    'BsmtFinType2': {'None': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6},
    'HeatingQC': {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
    'KitchenQual': {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
    'FireplaceQu': {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
    'GarageFinish': {'None': 0, 'Unf': 1, 'RFn': 2, 'Fin': 3},
    'GarageQual': {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
    'GarageCond': {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
    'PoolQC': {'None': 0, 'Fa': 1, 'TA': 2, 'Gd': 3, 'Ex': 4},
    'Fence': {'None': 0, 'MnWw': 1, 'GdWo': 2, 'MnPrv': 3, 'GdPrv': 4},
    'Functional': {'Sal': 0, 'Sev': 1, 'Maj2': 2, 'Maj1': 3, 'Mod': 4, 'Min2': 5, 'Min1': 6, 'Typ': 7},
    'PavedDrive': {'N': 0, 'P': 1, 'Y': 2}
}

for col, mapping in ordinal_cols.items():
    df_all[col] = df_all[col].map(mapping)

# 3. One-Hot Encoding cho các cột Categorical không có thứ tự
# Các cột này chỉ là các nhóm/loại khác nhau, không có thứ hạng
cat_cols = df_all.select_dtypes(include=['object', 'str']).columns

df_all = pd.get_dummies(df_all, columns=cat_cols, dtype=int)

# 4. Scaling dữ liệu bằng StandardScaler
scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df_all), columns=df_all.columns)

# Lưu dữ liệu trung gian (Tùy chọn)
os.makedirs('../../exps_/data/feature1', exist_ok=True)
df_scaled.to_csv('../../exps_/data/feature1/df_preprocessed.csv', index=False)

print("Hoàn thành Tiền xử lý. Kích thước df_scaled:", df_scaled.shape)

Hoàn thành Tiền xử lý. Kích thước df_scaled: (2919, 229)
